# 金融 CoT/PoT 可验证偏好优化：program_mixed_merged -> mixed DPO / mixed GRPO

本 notebook 直接承接 `run_fingpt_cot_pot.ipynb` 的 mixed Program SFT 结果，把

`/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged`

作为唯一 policy 起点，继续做后训练与 benchmark。

这里的目标不是回退到旧版“单一 schema RL”，而是明确采用单策略、双 schema、双 reward contract：

`program_mixed_merged -> DPO(strict Program preference optimization) / GRPO(mixed-policy RL)`

其中，GRPO 的 core 是 `program_numeric`，严格使用 `Evidence + Program`；supplement 是 `cot_answer_only`，只使用 `Reasoning + Answer`。

关键原则：

- 主 schema 继承 `run_fingpt_cot_pot.ipynb` 的 strict `program_executor_sft`。
- `program_numeric` 样本只学习 `Evidence + Program`，最终答案由 executor 计算。
- `cot_answer_only` supplement 只作为受控补充，不参与 strict Program 格式学习，也不进入主 benchmark 指标。
- DPO 和 GRPO 都保留，但它们必须共享同一套 mixed contract，避免 schema 冲突和 reward 失真。


## 框架选择

本 notebook 保留 **DPO** 和 **GRPO** 两条主线，但二者不再各自维护一套隐式 schema，而是共同遵守 mixed contract：

- **DPO**：只对 `program_numeric` 样本做 strict Program 偏好优化。chosen 是 gold `Evidence + Program`，rejected 是 program/schema mutation。DPO 不接入外部 CoT supplement。
- **GRPO**：继续使用 TRL `GRPOTrainer`，但 dataset 中显式区分 `program_numeric` 与 `cot_answer_only`。前者走可执行 reward 主项，后者只走 CoT-only reward。

推荐顺序：

1. 先构建 DPO / GRPO 数据，检查 schema。
2. 先跑 DPO smoke，确认 strict Program 输出没有被破坏。
3. 再跑 GRPO smoke，确认 reward contract 与 raw prompt 传递正常。
4. 最后比较 `program_mixed_merged`、DPO adapter、GRPO adapter 的 strict Program 主 benchmark。


In [ ]:
import json
import math
import os
import random
import re
import subprocess
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd

try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('cuda devices:', torch.cuda.device_count())
except Exception as exc:
    print('torch import failed:', repr(exc))

try:
    import datasets
    print('datasets:', datasets.__version__)
except Exception as exc:
    print('datasets import failed:', repr(exc))

try:
    import trl
    print('trl:', trl.__version__)
except Exception as exc:
    print('trl import failed. Install/update TRL before GRPO training:', repr(exc))


In [ ]:
BASE_MODEL = Path('/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct')
POLICY_BASE = Path('/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged')
SFT2_MERGED_OUT = POLICY_BASE  # kept for compatibility with older cells

RL_DATA_ROOT = Path('/root/autodl-tmp/data/financial_reasoning_cot_pot/rl')
FINO1_CACHE_DIR = RL_DATA_ROOT / 'fino1_finqa_path'
FINCOT_CACHE_DIR = RL_DATA_ROOT / 'fincot'
GRPO_TRAIN_FILE = RL_DATA_ROOT / 'train_cot_pot_grpo_mixed.jsonl'
GRPO_VALID_FILE = RL_DATA_ROOT / 'valid_cot_pot_grpo_mixed.jsonl'
GRPO_SMOKE_FILE = RL_DATA_ROOT / 'smoke_cot_pot_grpo_mixed.jsonl'

DPO_DATA_ROOT = Path('/root/autodl-tmp/data/financial_reasoning_cot_pot/dpo_pairs')
DPO_MIXED_FILE = DPO_DATA_ROOT / 'train_cot_pot_program_dpo.jsonl'
DPO_TRAIN_DIR = DPO_DATA_ROOT / 'train_dir'
DPO_SMOKE_FILE = DPO_DATA_ROOT / 'smoke_cot_pot_program_dpo.jsonl'
DPO_SMOKE_DIR = DPO_DATA_ROOT / 'smoke_dir'

OUTPUT_ROOT = Path('/root/autodl-tmp/outputs/financial_reasoning_cot_pot')
DPO_OUT = OUTPUT_ROOT / 'dpo_program_lora'
DPO_SMOKE_OUT = OUTPUT_ROOT / 'dpo_program_lora_smoke'
DPO_BENCH_OUT = OUTPUT_ROOT / 'benchmarks' / 'dpo_program_passk'
GRPO_OUT = OUTPUT_ROOT / 'grpo_cot_pot_lora'
GRPO_SMOKE_OUT = OUTPUT_ROOT / 'grpo_cot_pot_lora_smoke'
GRPO_BENCH_OUT = OUTPUT_ROOT / 'benchmarks' / 'grpo_cot_pot_passk'

PROGRAM_DATA_ROOT = Path('/root/autodl-tmp/data/financial_reasoning_cot_pot/program_sft')
FINQA_TRAIN_STRICT = PROGRAM_DATA_ROOT / 'train_finqa_program_strict.jsonl'
SFT2_CONV_STRICT = PROGRAM_DATA_ROOT / 'train_convfinqa_turn_program_strict.jsonl'
SFT2_FINQA_REPLAY = PROGRAM_DATA_ROOT / 'train_program_mixed.jsonl'  # optional; may not exist in early runs

# Fallbacks to existing v3 program data if cot_pot strict files have not been generated yet.
if not FINQA_TRAIN_STRICT.exists():
    FINQA_TRAIN_STRICT = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft1_program_strict.jsonl')
if not SFT2_CONV_STRICT.exists():
    SFT2_CONV_STRICT = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_convfinqa_turn_program_strict.jsonl')
if not SFT2_FINQA_REPLAY.exists():
    SFT2_FINQA_REPLAY = Path('/root/autodl-tmp/data/financial_reasoning_v3/clean/train_sft2_finqa_replay_program.jsonl')

FINQA_EVAL_FILE = Path('/root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json')
CONVFINQA_EVAL_FILE = Path('/root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json')
DESIGN_DOC = Path('/root/FinQA/docs/fin_pot_cot_rl.md')

for path in [
    RL_DATA_ROOT, FINO1_CACHE_DIR, FINCOT_CACHE_DIR,
    DPO_DATA_ROOT, DPO_TRAIN_DIR, DPO_SMOKE_DIR,
    DPO_OUT, DPO_SMOKE_OUT, DPO_BENCH_OUT,
    GRPO_OUT, GRPO_SMOKE_OUT, GRPO_BENCH_OUT,
]:
    path.mkdir(parents=True, exist_ok=True)

print('BASE_MODEL exists:', BASE_MODEL.exists(), BASE_MODEL)
print('POLICY_BASE exists:', POLICY_BASE.exists(), POLICY_BASE)
print('FINQA_TRAIN_STRICT exists:', FINQA_TRAIN_STRICT.exists(), FINQA_TRAIN_STRICT)
print('SFT2_CONV_STRICT exists:', SFT2_CONV_STRICT.exists(), SFT2_CONV_STRICT)
print('SFT2_FINQA_REPLAY exists:', SFT2_FINQA_REPLAY.exists(), SFT2_FINQA_REPLAY)
print('DESIGN_DOC exists:', DESIGN_DOC.exists(), DESIGN_DOC)
print('RL_DATA_ROOT:', RL_DATA_ROOT)


## 数据集方案

本 notebook 保留 mixed 路线，但明确拆成两种 reward profile：

1. **DPO strict-program pairs**
   - 来源：FinQA / ConvFinQA strict Program ShareGPT 数据。
   - `response_chosen`：gold `Evidence + Program`。
   - `response_rejected`：自动扰动 Program 或结构字段。
   - 不接入 Fino1/FinCoT supplement，避免 DPO 学到第二套 schema 偏好。

2. **GRPO mixed data**
   - core：FinQA / ConvFinQA strict Program 样本，`reward_profile = program_numeric`。
   - supplement：Fino1 / FinCoT 外部 CoT 样本，`reward_profile = cot_answer_only`。

统一数据字段：

- `input_prompt_raw`
- `reward_profile`
- `source_dataset`
- `gold_program`
- `gold_answer`
- `reference_response`
- `record_id`
- `metadata`

训练时再从 `input_prompt_raw` 动态构造 chat prompt，避免覆盖原始题面。所有 evidence-level reward 默认读取 `input_prompt_raw`，而不是 chat list 的字符串化结果。


## Schema Contract

这一节把 mixed notebook 的公共 contract 固定下来，避免后续数据构造、reward 和 benchmark 口径漂移。

- `program_numeric_output_schema = Evidence + Program`
- `cot_answer_only_output_schema = Reasoning + Answer`
- DPO / GRPO 都必须显式接收 `reward_profile`
- 主 benchmark 只报告 strict Program 指标；supplement 只做诊断，不混入主表


In [ ]:
PROGRAM_NUMERIC_REQUIRED = ['Evidence:', 'Program:']
PROGRAM_NUMERIC_FORBIDDEN = ['Reasoning:', 'Answer:', 'Normalized Answer:', 'Program: N/A']
COT_ANSWER_ONLY_REQUIRED = ['Reasoning:', 'Answer:']
COT_ANSWER_ONLY_FORBIDDEN = ['Program:', 'Normalized Answer:']
REWARD_PROFILES = ['program_numeric', 'cot_answer_only']
STRICT_PROGRAM_POLICY = Path('/root/autodl-tmp/outputs/financial_reasoning_cot_pot/program_mixed_merged')

assert POLICY_BASE == STRICT_PROGRAM_POLICY, (POLICY_BASE, STRICT_PROGRAM_POLICY)
assert POLICY_BASE.name == 'program_mixed_merged'

schema_contract = pd.DataFrame([
    {
        'reward_profile': 'program_numeric',
        'output_schema': 'Evidence + Program',
        'required': ', '.join(PROGRAM_NUMERIC_REQUIRED),
        'forbidden': ', '.join(PROGRAM_NUMERIC_FORBIDDEN),
        'main_benchmark': True,
    },
    {
        'reward_profile': 'cot_answer_only',
        'output_schema': 'Reasoning + Answer',
        'required': ', '.join(COT_ANSWER_ONLY_REQUIRED),
        'forbidden': ', '.join(COT_ANSWER_ONLY_FORBIDDEN),
        'main_benchmark': False,
    },
])
print('policy_base:', POLICY_BASE)
print('reward_profiles:', REWARD_PROFILES)
display(schema_contract)


In [ ]:
SEED = 42
MAX_CORE_RL_ROWS = 3500
MAX_FINO1_FINQA_ROWS = 750
MAX_FINCOT_RL_ROWS = 750
VALID_RATIO = 0.05
CORE_CONV_TO_FINQA_RATIO = 2.0

FINO1_FINQA_PATH_DATASET = 'TheFinAI/Fino1_Reasoning_Path_FinQA'
FINO1_FINQA_SPLIT = 'train'
FINCOT_DATASET = 'TheFinAI/FinCoT'
FINCOT_SPLIT = 'RL'
USE_FINO1_FINQA = True
USE_FINCOT = True

random.seed(SEED)
print({
    'MAX_CORE_RL_ROWS': MAX_CORE_RL_ROWS,
    'MAX_FINO1_FINQA_ROWS': MAX_FINO1_FINQA_ROWS,
    'MAX_FINCOT_RL_ROWS': MAX_FINCOT_RL_ROWS,
    'VALID_RATIO': VALID_RATIO,
    'CORE_CONV_TO_FINQA_RATIO': CORE_CONV_TO_FINQA_RATIO,
    'USE_FINO1_FINQA': USE_FINO1_FINQA,
    'USE_FINCOT': USE_FINCOT,
})


In [ ]:
def read_jsonl(path: Path, max_rows: Optional[int] = None) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        print('missing:', path)
        return rows
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_rows is not None and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            count += 1
    return count


def sample_rows(rows: List[Dict[str, Any]], n: Optional[int], seed: int = SEED) -> List[Dict[str, Any]]:
    rows = list(rows)
    rng = random.Random(seed)
    rng.shuffle(rows)
    if n is None or n < 0 or n >= len(rows):
        return rows
    return rows[:n]


def split_train_valid(rows: List[Dict[str, Any]], valid_ratio: float, seed: int = SEED) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    rows = sample_rows(rows, None, seed)
    valid_n = max(1, int(round(len(rows) * valid_ratio))) if rows else 0
    return rows[valid_n:], rows[:valid_n]


def first_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return ''.join(first_text(v) for v in value)
    if isinstance(value, dict):
        for key in ('text', 'content', 'value'):
            if key in value:
                return first_text(value[key])
        return json.dumps(value, ensure_ascii=False)
    return str(value)


In [ ]:
ANCHORS = ['Evidence:', 'Reasoning:', 'Program:', 'Answer:', 'Normalized Answer:']
PROGRAM_OPS = ['add', 'subtract', 'multiply', 'divide', 'greater', 'table_max', 'table_min', 'table_sum', 'table_average', 'average', 'sum', 'max', 'min']
NUMBER_RE = re.compile(r'-?\d+(?:,\d{3})*(?:\.\d+)?%?')


def extract_anchor(text: str, anchor: str) -> str:
    text = first_text(text)
    if not text or anchor not in text:
        return ''
    start = text.index(anchor) + len(anchor)
    end = len(text)
    for other in ANCHORS:
        if other == anchor:
            continue
        pos = text.find(other, start)
        if pos >= 0:
            end = min(end, pos)
    return text[start:end].strip()


def completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        if 'content' in completion:
            return first_text(completion.get('content'))
        if 'text' in completion:
            return first_text(completion.get('text'))
    if isinstance(completion, list) and completion:
        return completion_text(completion[0])
    return first_text(completion)


def normalize_number(text: str) -> Optional[float]:
    text = first_text(text)
    if not text:
        return None
    matches = NUMBER_RE.findall(text.replace(',', ''))
    if not matches:
        return None
    raw = matches[-1]
    is_percent = raw.endswith('%')
    if is_percent:
        raw = raw[:-1]
    try:
        value = float(raw)
    except ValueError:
        return None
    return value / 100.0 if is_percent else value


def numeric_equal(pred: str, gold: str, abs_tol: float = 1e-4, rel_tol: float = 1e-4) -> bool:
    # Contract: percentages like 10% are normalized to 0.10 before comparison.
    pred_num = normalize_number(pred)
    gold_num = normalize_number(gold)
    if pred_num is None or gold_num is None:
        return first_text(pred).strip().lower() == first_text(gold).strip().lower()
    return abs(pred_num - gold_num) <= max(abs_tol, abs(gold_num) * rel_tol)


def program_ops(program: str) -> List[str]:
    program = first_text(program).lower()
    return [op for op in PROGRAM_OPS if re.search(rf'\b{re.escape(op)}\s*\(', program)]


def prompt_text_from_any(value: Any) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict) and item.get('role') == 'user':
                parts.append(first_text(item.get('content')))
            else:
                parts.append(first_text(item))
        return '\n'.join(x for x in parts if x).strip()
    if isinstance(value, dict):
        return first_text(value.get('content') or value.get('text') or value.get('value'))
    return first_text(value)


In [ ]:
def sharegpt_to_grpo(row: Dict[str, Any], source_dataset: str) -> Optional[Dict[str, Any]]:
    conv = row.get('conversations') or []
    if len(conv) < 2:
        return None
    input_prompt_raw = conv[0].get('value') or conv[0].get('content') or ''
    target = conv[1].get('value') or conv[1].get('content') or ''
    if not input_prompt_raw or not target:
        return None
    meta = row.get('metadata') or {}
    gold_answer = meta.get('answer_norm') or meta.get('answer_exe') or extract_anchor(target, 'Normalized Answer:') or extract_anchor(target, 'Answer:')
    gold_program = meta.get('program_canonical') or meta.get('program_raw') or extract_anchor(target, 'Program:')
    if gold_answer is None or first_text(gold_answer) == '':
        return None
    return {
        'input_prompt_raw': input_prompt_raw,
        'gold_answer': first_text(gold_answer),
        'gold_program': first_text(gold_program),
        'reference_response': target,
        'source_dataset': source_dataset,
        'task_type': 'program_verifiable',
        'reward_profile': 'program_numeric',
        'record_id': first_text(row.get('record_id') or meta.get('record_id') or row.get('id') or meta.get('raw_id')),
        'metadata': {
            'source_path': source_dataset,
            'output_schema': 'Evidence + Program',
            'program_available': True,
        },
    }


finqa_candidates = []
for path in [FINQA_TRAIN_STRICT, SFT2_FINQA_REPLAY]:
    if path and Path(path).exists():
        finqa_candidates.extend([x for x in (sharegpt_to_grpo(r, 'finqa') for r in read_jsonl(Path(path))) if x])

conv_candidates = [x for x in (sharegpt_to_grpo(r, 'convfinqa_turn') for r in read_jsonl(SFT2_CONV_STRICT)) if x]


def dedupe_by_prompt(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for row in rows:
        key = row['input_prompt_raw']
        if key in seen:
            continue
        seen.add(key)
        out.append(row)
    return out


finqa_candidates = dedupe_by_prompt(finqa_candidates)
conv_candidates = dedupe_by_prompt(conv_candidates)

conv_target = int(round(MAX_CORE_RL_ROWS * CORE_CONV_TO_FINQA_RATIO / (CORE_CONV_TO_FINQA_RATIO + 1)))
finqa_target = MAX_CORE_RL_ROWS - conv_target
core_rows = sample_rows(conv_candidates, conv_target, SEED + 1) + sample_rows(finqa_candidates, finqa_target, SEED + 2)
core_rows = sample_rows(core_rows, None, SEED + 3)

print('FinQA candidates:', len(finqa_candidates), 'target:', finqa_target)
print('ConvFinQA candidates:', len(conv_candidates), 'target:', conv_target)
print('Core rows:', len(core_rows))
pd.DataFrame(core_rows[:3])[['source_dataset', 'reward_profile', 'gold_answer', 'gold_program']]


In [ ]:
def build_external_prompt(question: str) -> str:
    return (
        'You are a financial reasoning assistant. Solve the question with concise reasoning.\n\n'
        f'Question:\n{question.strip()}\n\n'
        'Output format:\n'
        'Reasoning: ...\n\n'
        'Answer: ...'
    )


def load_fino1_finqa_path(max_rows: int = MAX_FINO1_FINQA_ROWS) -> List[Dict[str, Any]]:
    if not USE_FINO1_FINQA:
        return []
    try:
        from datasets import load_dataset
        ds = load_dataset(FINO1_FINQA_PATH_DATASET, split=FINO1_FINQA_SPLIT, cache_dir=str(FINO1_CACHE_DIR))
    except Exception as exc:
        print('Could not load Fino1 Reasoning Path FinQA. Rerun with network access or set USE_FINO1_FINQA=False.')
        print(repr(exc))
        return []

    rows = []
    for rec in ds:
        question = first_text(rec.get('Open-ended Verifiable Question') or rec.get('Question') or rec.get('question'))
        gold_answer = first_text(rec.get('Ground-True Answer') or rec.get('Ground-Truth Answer') or rec.get('Answer') or rec.get('answer'))
        reasoning = first_text(rec.get('Complex_CoT') or rec.get('Complex CoT') or rec.get('Reasoning_process') or rec.get('reasoning'))
        response = first_text(rec.get('Response') or rec.get('Final_response') or rec.get('response'))
        if not question or not gold_answer:
            continue
        rows.append({
            'input_prompt_raw': build_external_prompt(question),
            'gold_answer': gold_answer,
            'gold_program': '',
            'reference_reasoning': reasoning,
            'reference_response': response,
            'negative_reasoning': '',
            'negative_response': '',
            'source_dataset': 'fino1_finqa_path',
            'task_type': 'cot_answer_only',
            'reward_profile': 'cot_answer_only',
            'record_id': first_text(rec.get('id') or rec.get('question_id')),
            'metadata': {
                'program_available': False,
                'output_schema': 'Reasoning + Answer',
            },
        })
    return sample_rows(rows, max_rows, SEED + 4)


def load_fincot_rl(max_rows: int = MAX_FINCOT_RL_ROWS) -> List[Dict[str, Any]]:
    if not USE_FINCOT:
        return []
    try:
        from datasets import load_dataset
        ds = load_dataset(FINCOT_DATASET, split=FINCOT_SPLIT, cache_dir=str(FINCOT_CACHE_DIR))
    except Exception as exc:
        print('Could not load FinCoT RL split. Rerun with network access or set USE_FINCOT=False.')
        print(repr(exc))
        return []

    rows = []
    for rec in ds:
        question = first_text(rec.get('Question'))
        gold_answer = first_text(rec.get('Answer') or rec.get('Final_response'))
        if not question or not gold_answer:
            continue
        rows.append({
            'input_prompt_raw': build_external_prompt(question),
            'gold_answer': gold_answer,
            'gold_program': '',
            'reference_reasoning': first_text(rec.get('Reasoning_process')),
            'reference_response': first_text(rec.get('Final_response')),
            'negative_reasoning': first_text(rec.get('Negative_reasoning_process')),
            'negative_response': first_text(rec.get('Negative_response')),
            'source_dataset': 'fincot_rl',
            'task_type': 'cot_answer_only',
            'reward_profile': 'cot_answer_only',
            'record_id': '',
            'metadata': {
                'program_available': False,
                'output_schema': 'Reasoning + Answer',
            },
        })
    return sample_rows(rows, max_rows, SEED + 5)


fino1_rows = load_fino1_finqa_path()
fincot_rows = load_fincot_rl()
print('Fino1 Reasoning Path FinQA rows:', len(fino1_rows))
print('FinCoT RL rows:', len(fincot_rows))
pd.DataFrame((fino1_rows + fincot_rows)[:5])[['source_dataset', 'reward_profile', 'gold_answer', 'gold_program']]


In [ ]:
print('External CoT sources:')
print('Fino1 dataset:', FINO1_FINQA_PATH_DATASET, 'split:', FINO1_FINQA_SPLIT, 'enabled:', USE_FINO1_FINQA)
print('FinCoT dataset:', FINCOT_DATASET, 'split:', FINCOT_SPLIT, 'enabled:', USE_FINCOT)
print('Local distill data is intentionally ignored in this notebook.')
print('Design doc:', DESIGN_DOC)

source_contract = pd.DataFrame([
    {'source': 'finqa/convfinqa_turn', 'reward_profile': 'program_numeric', 'program_reward': True, 'main_benchmark': True, 'role': 'strict Program / verifiable RL'},
    {'source': 'fino1_finqa_path', 'reward_profile': 'cot_answer_only', 'program_reward': False, 'main_benchmark': False, 'role': 'FinQA-style CoT supplement'},
    {'source': 'fincot_rl', 'reward_profile': 'cot_answer_only', 'program_reward': False, 'main_benchmark': False, 'role': 'broad financial CoT supplement'},
])

display(source_contract)
assert set(source_contract['reward_profile']) == {'program_numeric', 'cot_answer_only'}


In [ ]:
def audit_mixed_rows(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    bad_program_numeric = []
    bad_cot_prompt = []
    for i, row in enumerate(rows):
        profile = row.get('reward_profile')
        ref = first_text(row.get('reference_response'))
        raw_prompt = first_text(row.get('input_prompt_raw'))
        if profile == 'program_numeric':
            if any(anchor in ref for anchor in PROGRAM_NUMERIC_FORBIDDEN):
                bad_program_numeric.append(i)
        elif profile == 'cot_answer_only':
            if 'Program:' in raw_prompt or 'Program: N/A' in raw_prompt:
                bad_cot_prompt.append(i)
    return {
        'rows': len(rows),
        'bad_program_numeric_schema_rows': len(bad_program_numeric),
        'bad_cot_prompt_rows': len(bad_cot_prompt),
    }


mixed_rows = sample_rows(core_rows + fino1_rows + fincot_rows, None, SEED + 6)
train_rows, valid_rows = split_train_valid(mixed_rows, VALID_RATIO, SEED + 7)
smoke_rows = sample_rows(mixed_rows, 20, SEED + 8)

write_jsonl(GRPO_TRAIN_FILE, train_rows)
write_jsonl(GRPO_VALID_FILE, valid_rows)
write_jsonl(GRPO_SMOKE_FILE, smoke_rows)

summary = pd.DataFrame(mixed_rows).groupby(['source_dataset', 'reward_profile']).size().reset_index(name='rows') if mixed_rows else pd.DataFrame()
audit_summary = audit_mixed_rows(mixed_rows)
print('train:', len(train_rows), GRPO_TRAIN_FILE)
print('valid:', len(valid_rows), GRPO_VALID_FILE)
print('smoke:', len(smoke_rows), GRPO_SMOKE_FILE)
print('audit:', audit_summary)
display(summary)
preview_cols = ['source_dataset', 'reward_profile', 'gold_answer', 'gold_program', 'input_prompt_raw']
display(pd.DataFrame(smoke_rows[:5])[preview_cols])


## DPO 数据与训练

DPO 继续保留为主线之一，但它只服务 strict Program 路线，不参与 supplement mixed schema。

DPO contract：



外部 CoT supplement 不进入 DPO pair 构造。DPO 的目标是验证模型能否更稳定地偏好 strict  输出，而不是学习第二套  偏好。


In [ ]:
from financial_data_processors.common import build_rejected_from_strict_response

DPO_TOTAL_BUDGET = 5000
DPO_CONV_TO_FINQA_RATIO = 2.0
DPO_SMOKE_ROWS = 64


def sharegpt_to_dpo(row: Dict[str, Any], source_dataset: str) -> Optional[Dict[str, Any]]:
    conv = row.get('conversations') or []
    if len(conv) < 2:
        return None
    input_prompt_raw = conv[0].get('value') or conv[0].get('content') or ''
    chosen = conv[1].get('value') or conv[1].get('content') or ''
    if not input_prompt_raw or not chosen or 'Program:' not in chosen:
        return None
    rejected = build_rejected_from_strict_response(chosen)
    return {
        'system': '',
        'history': [],
        'question': input_prompt_raw,
        'input_prompt_raw': input_prompt_raw,
        'reward_profile': 'program_numeric',
        'response_chosen': chosen,
        'response_rejected': rejected,
        'source_dataset': source_dataset,
        'record_id': first_text(row.get('record_id') or (row.get('metadata') or {}).get('record_id')),
        'metadata': {
            'program_available': True,
            'base_policy': str(POLICY_BASE),
            'output_schema': 'Evidence + Program',
        },
    }


finqa_dpo = [x for x in (sharegpt_to_dpo(r, 'finqa') for r in read_jsonl(FINQA_TRAIN_STRICT)) if x]
conv_dpo = [x for x in (sharegpt_to_dpo(r, 'convfinqa_turn') for r in read_jsonl(SFT2_CONV_STRICT)) if x]

conv_target = int(round(DPO_TOTAL_BUDGET * DPO_CONV_TO_FINQA_RATIO / (DPO_CONV_TO_FINQA_RATIO + 1)))
finqa_target = DPO_TOTAL_BUDGET - conv_target
dpo_rows = sample_rows(conv_dpo, conv_target, SEED + 31) + sample_rows(finqa_dpo, finqa_target, SEED + 32)
dpo_rows = sample_rows(dpo_rows, None, SEED + 33)
dpo_smoke_rows = sample_rows(dpo_rows, DPO_SMOKE_ROWS, SEED + 34)

write_jsonl(DPO_MIXED_FILE, dpo_rows)
write_jsonl(DPO_SMOKE_FILE, dpo_smoke_rows)
(DPO_TRAIN_DIR / DPO_MIXED_FILE.name).write_text(DPO_MIXED_FILE.read_text(encoding='utf-8'), encoding='utf-8')
(DPO_SMOKE_DIR / DPO_SMOKE_FILE.name).write_text(DPO_SMOKE_FILE.read_text(encoding='utf-8'), encoding='utf-8')

print({
    'finqa_dpo_candidates': len(finqa_dpo),
    'conv_dpo_candidates': len(conv_dpo),
    'dpo_rows': len(dpo_rows),
    'dpo_file': str(DPO_MIXED_FILE),
    'dpo_smoke_rows': len(dpo_smoke_rows),
    'reward_profile': 'program_numeric',
})
display(pd.DataFrame(dpo_rows[:3])[['source_dataset', 'record_id', 'reward_profile', 'response_chosen', 'response_rejected']])


In [ ]:
DPO_MAX_STEPS = 120
DPO_SMOKE_MAX_STEPS = 1
DPO_LEARNING_RATE = '5e-6'
DPO_MAX_SOURCE_LENGTH = '1536'
DPO_MAX_TARGET_LENGTH = '256'
RUN_DPO_SMOKE = False
RUN_FULL_DPO = False


def dpo_command(train_dir: Path, output_dir: Path, max_steps: int) -> List[str]:
    return [
        'python', '-m', 'training.dpo_training',
        '--model_name_or_path', str(POLICY_BASE),
        '--tokenizer_name_or_path', str(BASE_MODEL),
        '--template_name', 'qwen',
        '--validation_split_percentage', '1',
        '--eval_strategy', 'no',
        '--train_file_dir', str(train_dir),
        '--do_train',
        '--use_peft', 'True',
        '--per_device_train_batch_size', '1',
        '--gradient_accumulation_steps', '16',
        '--gradient_checkpointing', 'True',
        '--learning_rate', DPO_LEARNING_RATE,
        '--max_steps', str(max_steps),
        '--max_source_length', DPO_MAX_SOURCE_LENGTH,
        '--max_target_length', DPO_MAX_TARGET_LENGTH,
        '--logging_steps', '10',
        '--save_steps', '40',
        '--logging_first_step', 'True',
        '--target_modules', 'q_proj,k_proj,v_proj,o_proj',
        '--lora_rank', '8',
        '--lora_alpha', '16',
        '--lora_dropout', '0.05',
        '--torch_dtype', 'bfloat16',
        '--device_map', 'auto',
        '--ddp_find_unused_parameters', 'False',
        '--output_dir', str(output_dir),
    ]

if RUN_DPO_SMOKE:
    cmd = dpo_command(DPO_SMOKE_DIR, DPO_SMOKE_OUT, DPO_SMOKE_MAX_STEPS)
    print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd='/root/FinQA')
elif RUN_FULL_DPO:
    cmd = dpo_command(DPO_TRAIN_DIR, DPO_OUT, DPO_MAX_STEPS)
    print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd='/root/FinQA')
else:
    print('DPO skipped. Set RUN_DPO_SMOKE=True first, then RUN_FULL_DPO=True for the full run.')


## Reward 函数

Reward contract 现在按 `reward_profile` 显式分流：

| reward_profile | 适用数据 | 主 schema | 主要 reward |
|---|---|---|---|
| `program_numeric` | FinQA / ConvFinQA | `Evidence + Program` | program execution、strict format、program op overlap、evidence grounding、轻量长度约束 |
| `cot_answer_only` | Fino1 / FinCoT | `Reasoning + Answer` | answer correctness、CoT format、简洁性、避免复述负样本回答 |

约束：

- `program_numeric` 不允许输出 `Reasoning:`、`Answer:`、`Normalized Answer:`。
- `cot_answer_only` 不要求 Program，也不应该被 Program reward 干扰。
- evidence-level reward 只读取 `input_prompt_raw`，不读取 chat list 的字符串化结果。
- 错误但可执行的 Program 不再获得保底正奖励。


In [ ]:
from evaluation.evaluate_financial_benchmarks import execute_prediction_program


def reward_answer(completions, gold_answer=None, reward_profile=None, **kwargs):
    rewards = []
    gold_answer = gold_answer or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, gold, profile in zip(completions, gold_answer, reward_profile):
        text = completion_text(completion)
        if profile == 'program_numeric':
            program = extract_anchor(text, 'Program:')
            executed_value, _, _ = execute_prediction_program(program)
            rewards.append(0.55 if executed_value is not None and numeric_equal(str(executed_value), gold) else 0.0)
        else:
            pred = extract_anchor(text, 'Answer:') or text
            rewards.append(0.60 if numeric_equal(pred, gold) else 0.0)
    return rewards


def reward_format(completions, reward_profile=None, **kwargs):
    rewards = []
    for completion, profile in zip(completions, reward_profile or ['program_numeric'] * len(completions)):
        text = completion_text(completion)
        if profile == 'program_numeric':
            required = PROGRAM_NUMERIC_REQUIRED
            forbidden = PROGRAM_NUMERIC_FORBIDDEN
            score = sum(1 for anchor in required if anchor in text) / len(required)
            penalty = sum(1 for anchor in forbidden if anchor in text) * 0.15
            rewards.append(max(0.0, score * 0.18 - penalty))
        else:
            required = COT_ANSWER_ONLY_REQUIRED
            forbidden = COT_ANSWER_ONLY_FORBIDDEN
            score = sum(1 for anchor in required if anchor in text) / len(required)
            penalty = sum(1 for anchor in forbidden if anchor in text) * 0.15
            rewards.append(max(0.0, score * 0.18 - penalty))
    return rewards


def reward_program(completions, gold_program=None, reward_profile=None, **kwargs):
    rewards = []
    gold_program = gold_program or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, gold, profile in zip(completions, gold_program, reward_profile):
        if profile != 'program_numeric':
            rewards.append(0.0)
            continue
        text = completion_text(completion)
        pred_program = extract_anchor(text, 'Program:')
        gold_ops = program_ops(gold)
        pred_ops = program_ops(pred_program)
        if not pred_program or pred_program.strip().upper() == 'N/A':
            rewards.append(0.0)
            continue
        executed_value, _, _ = execute_prediction_program(pred_program)
        parse_bonus = 0.08 if executed_value is not None else 0.0
        op_bonus = 0.0
        if gold_ops and pred_ops:
            op_bonus = sum(1 for op in set(gold_ops) if op in set(pred_ops)) / len(set(gold_ops)) * 0.12
        rewards.append(parse_bonus + op_bonus)
    return rewards


def reward_program_answer_consistency(completions, gold_answer=None, reward_profile=None, **kwargs):
    rewards = []
    gold_answer = gold_answer or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, gold, profile in zip(completions, gold_answer, reward_profile):
        if profile != 'program_numeric':
            rewards.append(0.0)
            continue
        program = extract_anchor(completion_text(completion), 'Program:')
        executed_value, _, _ = execute_prediction_program(program)
        if executed_value is None:
            rewards.append(0.0)
        elif numeric_equal(str(executed_value), gold):
            rewards.append(0.10)
        else:
            rewards.append(0.0)
    return rewards


def reward_evidence(completions, input_prompt_raw=None, prompt=None, reward_profile=None, **kwargs):
    rewards = []
    input_prompt_raw = input_prompt_raw or prompt or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, raw_prompt, profile in zip(completions, input_prompt_raw, reward_profile):
        if profile != 'program_numeric':
            rewards.append(0.0)
            continue
        evidence = extract_anchor(completion_text(completion), 'Evidence:')
        raw_prompt_text = prompt_text_from_any(raw_prompt)
        ev_nums = set(NUMBER_RE.findall(evidence.replace(',', '')))
        prompt_nums = set(NUMBER_RE.findall(raw_prompt_text.replace(',', '')))
        rewards.append(0.05 if ev_nums and ev_nums.intersection(prompt_nums) else 0.0)
    return rewards


def reward_brevity_and_relevance(completions, negative_response=None, reward_profile=None, **kwargs):
    rewards = []
    negative_response = negative_response or [''] * len(completions)
    reward_profile = reward_profile or ['program_numeric'] * len(completions)
    for completion, neg, profile in zip(completions, negative_response, reward_profile):
        text = completion_text(completion)
        words = text.split()
        if profile == 'program_numeric':
            rewards.append(0.05 if 8 <= len(words) <= 120 else 0.0)
        else:
            score = 0.0
            if 25 <= len(words) <= 220:
                score += 0.15
            neg_text = first_text(neg).strip()
            if neg_text and neg_text[:80] not in text:
                score += 0.10
            elif not neg_text:
                score += 0.05
            rewards.append(score)
    return rewards


REWARD_FUNCS = [
    reward_answer,
    reward_format,
    reward_program,
    reward_program_answer_consistency,
    reward_evidence,
    reward_brevity_and_relevance,
]
print([fn.__name__ for fn in REWARD_FUNCS])


In [ ]:
smoke_completions = [
    'Evidence:\n- the 2007 value is 991.1 and the 2008 value is 959.2\n- compute the year-over-year change\n\nProgram: divide(subtract(959.2, 991.1), 991.1)',
    'Evidence:\n- the values are listed above\n\nProgram: add(1, 1)',
    'Reasoning:\nThe answer follows directly from the provided financial path.\n\nAnswer: 10%',
    'Reasoning:\nThe answer is 10%.\n\nProgram: add(1,1)\n\nAnswer: 10%',
    'Evidence:\n- revenue was 20\n- cost was 15\n\nProgram: subtract(20, 15)\nadd(0, 0)',
]
smoke_kwargs = {
    'gold_answer': ['-0.03219', '-0.03219', '0.10', '0.10', '5'],
    'gold_program': [
        'divide(subtract(959.2, 991.1), 991.1)',
        'divide(subtract(959.2, 991.1), 991.1)',
        '',
        '',
        'subtract(20, 15)',
    ],
    'reward_profile': ['program_numeric', 'program_numeric', 'cot_answer_only', 'cot_answer_only', 'program_numeric'],
    'input_prompt_raw': [
        '2007 991.1 2008 959.2',
        '2007 991.1 2008 959.2',
        'Question: return 10%',
        'Question: return 10%',
        'Revenue 20 cost 15',
    ],
    'negative_response': ['', '', 'wrong answer text', 'wrong answer text', ''],
}
for fn in REWARD_FUNCS:
    print(fn.__name__, fn(smoke_completions, **smoke_kwargs))


## GRPO 训练

GRPO 继续保留 mixed-policy 路线，但 contract 已经固定：

- `program_numeric` 走 strict `Evidence + Program` reward 主项。
- `cot_answer_only` 只走 CoT-only reward，不参与 Program reward。
- 训练集必须保留 `input_prompt_raw`，训练时再动态构造 chat prompt。

下面的训练 cell 默认依然安全：`RUN_GRPO_SMOKE` 和 `RUN_FULL_GRPO` 都是 `False`。请先跑 smoke，确认 reward smoke test 与 raw prompt 传递都正常后，再启动完整训练。


In [ ]:
GRPO_NUM_GENERATIONS = 4
MAX_PROMPT_LENGTH = 1536
MAX_COMPLETION_LENGTH = 384
GRPO_LEARNING_RATE = 5e-6
BETA = 0.001
MAX_STEPS = 300
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
USE_VLLM = False

RUN_GRPO_SMOKE = False
RUN_FULL_GRPO = False

print({
    'GRPO_NUM_GENERATIONS': GRPO_NUM_GENERATIONS,
    'MAX_PROMPT_LENGTH': MAX_PROMPT_LENGTH,
    'MAX_COMPLETION_LENGTH': MAX_COMPLETION_LENGTH,
    'GRPO_LEARNING_RATE': GRPO_LEARNING_RATE,
    'BETA': BETA,
    'MAX_STEPS': MAX_STEPS,
    'USE_VLLM': USE_VLLM,
})


In [ ]:
def run_trl_grpo(train_file: Path, valid_file: Path, output_dir: Path, max_steps: int):
    from datasets import load_dataset
    from peft import LoraConfig
    from transformers import AutoTokenizer
    from trl import GRPOConfig, GRPOTrainer

    tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL), trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_ds = load_dataset('json', data_files=str(train_file), split='train')
    eval_ds = load_dataset('json', data_files=str(valid_file), split='train') if valid_file.exists() else None

    def to_chat_prompt(example):
        input_prompt_raw = example['input_prompt_raw']
        input_prompt_chat = [
            {'role': 'system', 'content': 'You are a financial numerical reasoning assistant. Follow the requested schema exactly.'},
            {'role': 'user', 'content': input_prompt_raw},
        ]
        return {
            'prompt': input_prompt_chat,
            'input_prompt_chat': input_prompt_chat,
        }

    train_ds = train_ds.map(to_chat_prompt)
    if eval_ds is not None:
        eval_ds = eval_ds.map(to_chat_prompt)

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    )

    args = GRPOConfig(
        output_dir=str(output_dir),
        learning_rate=GRPO_LEARNING_RATE,
        beta=BETA,
        max_steps=max_steps,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_generations=GRPO_NUM_GENERATIONS,
        max_prompt_length=MAX_PROMPT_LENGTH,
        max_completion_length=MAX_COMPLETION_LENGTH,
        logging_steps=10,
        save_steps=100,
        eval_strategy='steps' if eval_ds is not None else 'no',
        eval_steps=50,
        bf16=True,
        report_to='tensorboard',
        remove_unused_columns=False,
        use_vllm=USE_VLLM,
        log_completions=True,
    )

    trainer = GRPOTrainer(
        model=str(POLICY_BASE),
        reward_funcs=REWARD_FUNCS,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    return trainer


if RUN_GRPO_SMOKE:
    smoke_out = GRPO_SMOKE_OUT
    trainer = run_trl_grpo(GRPO_SMOKE_FILE, GRPO_SMOKE_FILE, smoke_out, max_steps=1)
elif RUN_FULL_GRPO:
    trainer = run_trl_grpo(GRPO_TRAIN_FILE, GRPO_VALID_FILE, GRPO_OUT, max_steps=MAX_STEPS)
else:
    print('Training skipped. Set RUN_GRPO_SMOKE=True first, then RUN_FULL_GRPO=True for the full run.')


## Benchmark 命令

主 benchmark 继续严格对齐 `run_fingpt_cot_pot.ipynb`：

- policy 起点 = `program_mixed_merged`
- 评测口径 = strict Program
- processor = `program_executor_sft`
- numeric output format = `cot_program`

注意：

- DPO / GRPO 主表只报告 FinQA / ConvFinQA strict Program 指标。
- CoT supplement 不进入主 benchmark。
- supplement 只在数据构成与诊断分析里出现，不与 strict Program 主指标混算。


In [ ]:
RUN_BENCHMARK_DPO = False
RUN_BENCHMARK_GRPO = False
RUN_BENCHMARK_SUPPLEMENT_DIAG = False


def benchmark_command(run_name: str, adapter_path: Path, output_dir: Path) -> List[str]:
    return [
        'python', '-m', 'evaluation.evaluate_financial_benchmarks',
        '--tokenizer_path', str(BASE_MODEL),
        '--model_entry', f'sft={POLICY_BASE}',
        '--model_entry', f'{run_name}={POLICY_BASE}',
        '--adapter_entry', f'{run_name}={adapter_path}',
        '--finqa_test_file', str(FINQA_EVAL_FILE),
        '--convfinqa_test_file', str(CONVFINQA_EVAL_FILE),
        '--finqa_max_samples', '100',
        '--convfinqa_max_samples', '100',
        '--max_new_tokens', '1024',
        '--pass_k', '1,4,8',
        '--num_samples_per_example', '8',
        '--sample_temperature', '0.7',
        '--sample_top_p', '0.95',
        '--sample_seed', '42',
        '--processor_sft_variant', 'program_executor_sft',
        '--numeric_output_format', 'cot_program',
        '--output_dir', str(output_dir),
    ]

for name, enabled, adapter, out_dir in [
    ('dpo', RUN_BENCHMARK_DPO, DPO_OUT, DPO_BENCH_OUT),
    ('grpo', RUN_BENCHMARK_GRPO, GRPO_OUT, GRPO_BENCH_OUT),
]:
    cmd = benchmark_command(name, adapter, out_dir)
    print(' '.join(cmd))
    if enabled:
        subprocess.run(cmd, check=True, cwd='/root/FinQA')
    else:
        print(f'{name} benchmark skipped. Set RUN_BENCHMARK_{name.upper()}=True after adapter exists.')

if RUN_BENCHMARK_SUPPLEMENT_DIAG:
    print('Supplement diagnostics are intentionally kept out of the strict Program main benchmark table.')
else:
    print('Supplement diag skipped. Keep main benchmark strict-program only.')


## 结果分析

这一节把 strict Program 主 benchmark 与 mixed supplement 诊断拆开看。

主分析目标：

- ：FinQA / ConvFinQA 上的 strict Program 指标是否提升。
- ：本次训练的数据组成是否发生明显漂移。
- ：supplement 只做诊断，不与主指标直接比较。


In [ ]:
SUMMARY_FILES = {
    'cot_pot_sft_passk': Path('/root/autodl-tmp/outputs/financial_reasoning_cot_pot/benchmarks/cot_pot_program_passk/benchmark_summary.csv'),
    'v2_sft2_passk': Path('/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/sft2_merged_passk/benchmark_summary.csv'),
    'v3_sft2_passk': Path('/root/autodl-tmp/outputs/financial_reasoning_v3/benchmarks/sft2_merged_passk/benchmark_summary.csv'),
    'dpo_program_passk': DPO_BENCH_OUT / 'benchmark_summary.csv',
    'grpo_cot_pot_passk': GRPO_BENCH_OUT / 'benchmark_summary.csv',
}


def reward_profile_mix_summary(path: Path) -> Dict[str, int]:
    rows = read_jsonl(path)
    return {
        'program_numeric_rows': sum(1 for row in rows if row.get('reward_profile') == 'program_numeric'),
        'cot_answer_only_rows': sum(1 for row in rows if row.get('reward_profile') == 'cot_answer_only'),
    }

frames = []
for run_name, path in SUMMARY_FILES.items():
    if not path.exists():
        print('missing summary:', run_name, path)
        continue
    df = pd.read_csv(path)
    df.insert(0, 'run_name', run_name)
    df['strict_program_main_metric'] = 'executed_answer_accuracy'
    df['supplement_diag_metric'] = 'not_in_main_benchmark'
    frames.append(df)

mix_counts = reward_profile_mix_summary(GRPO_TRAIN_FILE) if GRPO_TRAIN_FILE.exists() else {'program_numeric_rows': 0, 'cot_answer_only_rows': 0}
print('reward_profile_mix:', mix_counts)

if frames:
    all_summary = pd.concat(frames, ignore_index=True)
    all_summary['reward_profile_mix'] = f"program_numeric={mix_counts['program_numeric_rows']};cot_answer_only={mix_counts['cot_answer_only_rows']}"
    all_summary['program_numeric_rows'] = mix_counts['program_numeric_rows']
    all_summary['cot_answer_only_rows'] = mix_counts['cot_answer_only_rows']
    display(all_summary)
else:
    print('No benchmark summaries available yet.')


In [ ]:
def load_greedy_predictions(path: Path) -> Dict[Tuple[str, str], Dict[str, Any]]:
    rows = {}
    if not path.exists():
        return rows
    for row in read_jsonl(path):
        if row.get('generation_mode') == 'greedy':
            rows[(row.get('task_name', ''), row.get('record_id', ''))] = row
    return rows

sft2_preds = load_greedy_predictions(Path('/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/sft2_merged_passk/sft2_predictions.jsonl'))
grpo_preds = load_greedy_predictions(GRPO_BENCH_OUT / 'grpo_predictions.jsonl')
common_keys = sorted(set(sft2_preds).intersection(grpo_preds))
print('common greedy predictions:', len(common_keys))
for key in common_keys[:5]:
    sft2 = sft2_preds[key]
    grpo = grpo_preds[key]
    print('\n', key)
    print('sft2 correct:', sft2.get('answer_correct'), 'grpo correct:', grpo.get('answer_correct'))
    print('gold:', sft2.get('gold_answer'))
    print('sft2:', first_text(sft2.get('prediction'))[:500])
    print('grpo:', first_text(grpo.get('prediction'))[:500])


## 扩展说明

如果 DPO 相比 `program_mixed_merged` 没有提升，或 strict Program execution accuracy 下降，应停止扩大 DPO；这说明偏好数据只学到了格式扰动，而没有提升真实 Program reasoning。

如果 GRPO 有提升，后续优先扩展同一套 mixed contract：

- 用 pass@k mining 选择 hard-but-verifiable 的 strict Program 样本，而不是随机 SFT 样本。
- 继续收紧 strict DSL executor，避免自然语言 program 被误执行。
- 将 `program_numeric` 主 reward 继续绑定到 `execute(Program) == gold_answer`。
- supplement 只作为受控 CoT 补充，避免重新污染 strict Program schema。

Handoff back to SFT：

- 如果 GRPO 把 strict Program 主指标拉低，应先回看 mixed schema 权重与 reward contract。
- 不要首先怀疑 `run_fingpt_cot_pot.ipynb` 产出的 `program_mixed_merged` policy；它仍然是本 notebook 的唯一上游 policy。

工程扩展可以再考虑 verl / OpenRLHF；当前 notebook 先保留 TRL 单机路径，用于验证 mixed reward contract。
